[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/07_Observability_and_State_Estimation.ipynb)

# DiveLab

## Notebook 07 — Observability and State Estimation

**Guiding question:** If we measure only depth, can we reconstruct the diver's hidden state?

*Not every state must be measured directly — but the system must reveal enough information through its outputs.*

## Learning objectives

By the end of this lab, you will be able to:

- distinguish state variables from measured outputs;
- define observability for a linear state-space model;
- build the observability matrix;
- test observability using matrix rank;
- understand why depth history can reveal vertical velocity;
- construct a simple Luenberger observer;
- interpret observer error dynamics;
- connect deterministic observers with the Kalman filter.

## From Notebook 06 to Notebook 07

Notebook 06 introduced three different quantities:

$$
x(t)
$$

the true state,

$$
y(t)
$$

the measurement,

and:

$$
\hat x(t)
$$

the estimate.

A controller often needs the full state, but sensors may measure only part of it.

For example, suppose we measure depth but not vertical velocity.

Then the natural question is:

> Can velocity be reconstructed from the history of depth measurements and the system model?

## State-space model

We start from a linearized two-state model around an equilibrium:

$$
x=
\begin{bmatrix}
\delta z\\
\delta v
\end{bmatrix}
$$

with:

$$
\dot x=Ax
$$

and output:

$$
y=Cx
$$

If the sensor measures only depth:

$$
y=\delta z
$$

so:

$$
C=
\begin{bmatrix}
1 & 0
\end{bmatrix}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Linearized buoyancy model

From Notebook 03, near the equilibrium:

$$
\delta\dot z=-\delta v
$$

and:

$$
\delta\dot v=a_z\delta z
$$

where:

$$
a_z<0
$$

because buoyancy decreases as depth increases.

We use the same physical parameters as before.

In [ ]:
rho = 1025.0
g = 9.80665
P0 = 101325.0

mass = 90.0
z_e = 20.0
gas_surface_volume_e = 0.005

def pressure_at_depth(z):
    return P0 + rho * g * z

dFb_dz = (
    -rho * g
    * gas_surface_volume_e
    * P0
    * rho * g
    / pressure_at_depth(z_e)**2
)

a_z = dFb_dz / mass

A = np.array([
    [0.0, -1.0],
    [a_z, 0.0]
])

C = np.array([
    [1.0, 0.0]
])

print("A =")
print(A)
print()
print("C =")
print(C)

## What does observability mean?

A system is **observable** if the internal state can be reconstructed from the measured output over a finite time interval, assuming the model is known.

For:

$$
\dot x=Ax
$$

$$
y=Cx
$$

the observability matrix is:

$$
\mathcal O=
\begin{bmatrix}
C\\
CA\\
CA^2\\
\vdots\\
CA^{n-1}
\end{bmatrix}
$$

For an $n$-state system, the system is observable if:

$$
\operatorname{rank}(\mathcal O)=n
$$

## Build the observability matrix

Our model has two states, so:

$$
\mathcal O=
\begin{bmatrix}
C\\
CA
\end{bmatrix}
$$

In [ ]:
O = np.vstack([
    C,
    C @ A
])

print("Observability matrix:")
print(O)
print()
print("Rank =", np.linalg.matrix_rank(O))

## Interpretation

Because the rank is 2, the system is observable.

This means:

> measuring depth over time contains enough information, in principle, to reconstruct both depth and vertical velocity.

Why?

Because:

$$
y=\delta z
$$

and:

$$
\dot y=\delta\dot z=-\delta v
$$

So the time evolution of the measured depth reveals information about velocity.

## A useful caution

Observable does **not** mean easy to estimate in practice.

A system can be observable and still be difficult to estimate when:

- measurement noise is large;
- the model is inaccurate;
- sampling is poor;
- the system is weakly observable;
- sensor failures occur.

Observability is a structural property.

Estimation quality is an engineering problem.

# Part 1 — Simulate the true linear system

In [ ]:
def simulate_linear_system(
    x0,
    duration=20.0,
    dt=0.01
):
    n = int(duration / dt) + 1
    t = np.linspace(0, duration, n)

    x = np.zeros((n, 2))
    y = np.zeros(n)

    x[0] = x0
    y[0] = (C @ x[0])[0]

    for k in range(n - 1):
        dx = A @ x[k]
        x[k + 1] = x[k] + dx * dt
        y[k + 1] = (C @ x[k + 1])[0]

    return t, x, y

In [ ]:
x0 = np.array([0.0, 0.05])

t, x_true, y_true = simulate_linear_system(x0)

In [ ]:
plt.plot(t, x_true[:, 0], label="Depth deviation δz")
plt.plot(t, x_true[:, 1], label="Velocity deviation δv")

plt.xlabel("Time [s]")
plt.ylabel("State")
plt.title("True linearized state")
plt.grid(True)
plt.legend()
plt.show()

# Part 2 — Add depth-sensor noise

The sensor measures:

$$
y_m = Cx+n
$$

In [ ]:
rng = np.random.default_rng(42)

sigma_y = 0.03
noise = rng.normal(0.0, sigma_y, size=len(y_true))

y_meas = y_true + noise

In [ ]:
plt.plot(t, y_true, label="True depth deviation")
plt.plot(t, y_meas, alpha=0.5, label="Measured depth")

plt.xlabel("Time [s]")
plt.ylabel("Depth deviation [m]")
plt.title("Noisy measured output")
plt.grid(True)
plt.legend()
plt.show()

## Why not just differentiate the measurement?

Since:

$$
\delta v=-\dot y
$$

we might estimate velocity with a numerical derivative.

But Notebook 06 showed the problem:

> differentiation amplifies measurement noise.

So instead of differentiating the signal directly, we can combine:

- a model;
- the measurement;
- a correction term.

That leads to an **observer**.

# Part 3 — Luenberger observer

A continuous-time observer is:

$$
\dot{\hat x}
=
A\hat x
+
L(y-C\hat x)
$$

where:

$$
y-C\hat x
$$

is the **innovation** or estimation residual.

The observer predicts the state using the model, then corrects that prediction using the measurement error.

## Anatomy of the observer

The term:

$$
A\hat x
$$

is the model-based prediction.

The term:

$$
L(y-C\hat x)
$$

is the measurement correction.

So the observer behaves like:

> predict → compare → correct

## Estimation error dynamics

Define:

$$
e=x-\hat x
$$

For the ideal deterministic model:

$$
\dot e=(A-LC)e
$$

Therefore, the observer converges if the eigenvalues of:

$$
A-LC
$$

have negative real part.

## Choose observer poles

Because the pair $(A,C)$ is observable, we can choose $L$ to place the observer poles.

For our two-state system, let:

$$
L=
\begin{bmatrix}
l_1\\
l_2
\end{bmatrix}
$$

Then:

$$
A-LC
=
\begin{bmatrix}
-l_1 & -1\\
a_z-l_2 & 0
\end{bmatrix}
$$

We choose the observer dynamics faster than the plant dynamics, but not excessively fast because high observer gains amplify measurement noise.

In [ ]:
# Desired observer poles
p1 = -1.2
p2 = -1.8

# Characteristic polynomial:
# s^2 + l1*s + (l2 - a_z) = 0
l1 = -(p1 + p2)
l2 = p1 * p2 + a_z

L = np.array([
    [l1],
    [l2]
])

print("Observer gain L =")
print(L)
print()
print("Observer eigenvalues:")
print(np.linalg.eigvals(A - L @ C))

# Part 4 — Simulate the observer

In [ ]:
def simulate_observer(
    y_meas,
    xhat0,
    dt
):
    n = len(y_meas)
    xhat = np.zeros((n, 2))
    innovation = np.zeros(n)

    xhat[0] = xhat0

    for k in range(n - 1):
        yhat = (C @ xhat[k])[0]
        innovation[k] = y_meas[k] - yhat

        dxhat = (
            A @ xhat[k]
            + (L.flatten() * innovation[k])
        )

        xhat[k + 1] = xhat[k] + dxhat * dt

    innovation[-1] = y_meas[-1] - (C @ xhat[-1])[0]

    return xhat, innovation

In [ ]:
dt = t[1] - t[0]

xhat0 = np.array([0.3, -0.15])

x_hat, innovation = simulate_observer(
    y_meas=y_meas,
    xhat0=xhat0,
    dt=dt
)

In [ ]:
plt.plot(t, x_true[:, 0], label="True δz")
plt.plot(t, x_hat[:, 0], label="Estimated δz")

plt.xlabel("Time [s]")
plt.ylabel("Depth deviation [m]")
plt.title("Observer depth estimate")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.plot(t, x_true[:, 1], label="True δv")
plt.plot(t, x_hat[:, 1], label="Estimated δv")

plt.xlabel("Time [s]")
plt.ylabel("Velocity deviation [m/s]")
plt.title("Observer reconstructs unmeasured velocity")
plt.grid(True)
plt.legend()
plt.show()

## Key result

The observer estimates velocity even though velocity is never measured directly.

This is possible because:

- the system dynamics couple depth and velocity;
- the measured depth contains information about both states;
- the system is observable.

# Part 5 — Estimation error

In [ ]:
error = x_true - x_hat
error_norm = np.linalg.norm(error, axis=1)

plt.plot(t, error[:, 0], label="Depth estimation error")
plt.plot(t, error[:, 1], label="Velocity estimation error")

plt.xlabel("Time [s]")
plt.ylabel("Error")
plt.title("Observer estimation error")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.plot(t, error_norm)

plt.xlabel("Time [s]")
plt.ylabel("||e||")
plt.title("Estimation-error norm")
plt.grid(True)
plt.show()

If the observer poles are stable, the deterministic estimation error tends to decay.

With measurement noise, the estimate will not become perfectly exact.

Instead, it fluctuates around the true state.

# Part 6 — Innovation

The innovation is:

$$
r=y-C\hat x
$$

It tells us how different the measurement is from what the observer predicted.

In [ ]:
plt.plot(t, innovation)

plt.xlabel("Time [s]")
plt.ylabel("Innovation")
plt.title("Observer innovation")
plt.grid(True)
plt.show()

The innovation is central to estimation theory.

It can be used for:

- state correction;
- sensor-health monitoring;
- fault detection;
- statistical consistency checks.

This connects directly with the residual-based fault detection proposed in Notebook 06.

# Part 7 — Observer gain tradeoff

Faster observer poles usually produce faster convergence.

But faster observers also react more strongly to measurement noise.

Let's compare several observer designs.

In [ ]:
observer_pole_sets = [
    (-0.4, -0.7),
    (-1.2, -1.8),
    (-4.0, -5.0),
]

estimates = []

for q1, q2 in observer_pole_sets:
    l1_q = -(q1 + q2)
    l2_q = q1 * q2 + a_z

    L_q = np.array([
        [l1_q],
        [l2_q]
    ])

    xhat_q = np.zeros_like(x_true)
    xhat_q[0] = xhat0

    for k in range(len(t) - 1):
        resid = y_meas[k] - (C @ xhat_q[k])[0]
        dxhat = A @ xhat_q[k] + L_q.flatten() * resid
        xhat_q[k + 1] = xhat_q[k] + dxhat * dt

    estimates.append((q1, q2, xhat_q))

In [ ]:
plt.plot(t, x_true[:, 1], label="True velocity")

for q1, q2, xhat_q in estimates:
    plt.plot(t, xhat_q[:, 1], label=f"Observer poles {q1}, {q2}")

plt.xlabel("Time [s]")
plt.ylabel("Velocity deviation [m/s]")
plt.title("Observer speed vs noise sensitivity")
plt.grid(True)
plt.legend()
plt.show()

## Interpretation

Slow observer:

- less sensitive to noise;
- slower convergence.

Fast observer:

- quicker convergence;
- more noise amplification.

So observer design has the same kind of tradeoff seen previously:

> responsiveness vs robustness to imperfect information.

# Part 8 — What if the system is not observable?

Consider a deliberately modified output:

$$
C=
\begin{bmatrix}
0 & 0
\end{bmatrix}
$$

Then the sensor measures nothing about the state.

In [ ]:
C_bad = np.array([
    [0.0, 0.0]
])

O_bad = np.vstack([
    C_bad,
    C_bad @ A
])

print("Bad observability matrix:")
print(O_bad)
print()
print("Rank =", np.linalg.matrix_rank(O_bad))

The rank collapses.

No observer can reconstruct the state from an output that contains no information about it.

This illustrates an important distinction:

> estimation algorithms cannot create information that the sensors and dynamics do not provide.

# Part 9 — Connection to output-feedback control

With full-state feedback:

$$
u=-Kx
$$

the controller assumes the state is known.

If only the output is measured, we can instead use:

$$
u=-K\hat x
$$

where $\hat x$ comes from the observer.

This architecture is:

> plant → sensor → observer → controller → actuator → plant

## Separation principle preview

For linear systems under standard assumptions, controller design and observer design can often be treated separately:

- choose $K$ to stabilize the plant;
- choose $L$ to stabilize the observer error.

This is the **separation principle**.

It is one of the most elegant results in linear control theory.

# Part 10 — From deterministic observer to Kalman filter

The Luenberger observer assumes a deterministic model and uses a gain $L$ chosen by design.

The Kalman filter adds a probabilistic description of uncertainty.

A common discrete model is:

$$
x_{k+1}=Ax_k+Bu_k+w_k
$$

$$
y_k=Cx_k+v_k
$$

where:

- $w_k$ is process noise;
- $v_k$ is measurement noise.

## Process noise vs measurement noise

Measurement noise represents uncertainty in the sensor:

$$
v_k
$$

Process noise represents uncertainty in the plant model:

$$
w_k
$$

Examples of process uncertainty in our simplified diver model might include:

- unmodeled fin motion;
- breathing-induced buoyancy variation;
- current disturbances;
- model error in drag;
- small unmodeled gas-volume changes.

## The Kalman idea

The Kalman filter repeatedly performs two conceptual steps:

### Predict

Use the model to predict:

$$
\hat x_{k|k-1}
$$

and its uncertainty.

### Correct

Compare the measurement with the prediction:

$$
r_k=y_k-C\hat x_{k|k-1}
$$

and correct the estimate according to how much the model and sensor are trusted.

The weighting is computed automatically from uncertainty models.

## Why this is a natural next step

The Luenberger observer taught us:

> model prediction + measurement correction.

The Kalman filter keeps the same structure but chooses the correction gain using covariance information.

So conceptually:

$$
\text{Luenberger observer}
\quad\longrightarrow\quad
\text{Kalman filter}
$$

is a very natural progression.

## Optional numerical preview: complementary estimation

Before implementing a full Kalman filter, we can show a simple weighted correction:

$$
\hat z
=
(1-\beta)\hat z_{\text{model}}
+
\beta z_m
$$

Small $\beta$ trusts the model more.

Large $\beta$ trusts the sensor more.

In [ ]:
beta_values = [0.1, 0.5, 0.9]

model_prediction = y_true

for beta in beta_values:
    z_blend = (1 - beta) * model_prediction + beta * y_meas
    plt.plot(t, z_blend, label=f"beta = {beta}")

plt.plot(t, y_true, "--", label="True output")

plt.xlabel("Time [s]")
plt.ylabel("Depth deviation [m]")
plt.title("Simple model-sensor blending")
plt.grid(True)
plt.legend()
plt.show()

This is not a Kalman filter.

But it illustrates the same basic question:

> how much should we trust the model, and how much should we trust the sensor?

## Systems-theory connections

Notebook 07 introduces several core concepts.

### Observability

Can the internal state be reconstructed from outputs?

### Observer

A dynamical system that estimates the state.

### Innovation

Difference between measured and predicted output.

### Observer poles

They determine the convergence speed of the estimation error.

### Output feedback

The controller uses an estimated state instead of the true state.

### Separation principle

Controller and observer can often be designed independently in linear systems.

### Kalman filtering

Optimal linear state estimation under standard stochastic assumptions.

## Exercises

### 1. Check observability at another equilibrium

Change:

```python
z_e
```

and recompute:

- $A$;
- the observability matrix;
- its rank.

Does observability change?

### 2. Change the measured output

Try:

```python
C = np.array([[0.0, 1.0]])
```

which measures only velocity.

Is the system observable?

### 3. Observer speed

Try observer poles:

```python
(-0.3, -0.5)
(-2.0, -3.0)
(-8.0, -10.0)
```

Compare convergence and noise sensitivity.

### 4. Wrong initial estimate

Try:

```python
xhat0 = np.array([1.0, -0.5])
```

Does the observer recover the true state?

### 5. Increase measurement noise

Increase:

```python
sigma_y
```

How does the velocity estimate change?

## Challenge — output-feedback controller

Combine:

- a state-feedback law;
- the Luenberger observer.

Use:

$$
u=-K\hat x
$$

instead of $u=-Kx$.

Compare:

- full-state feedback;
- observer-based feedback;
- observer-based feedback with measurement noise.

In [ ]:
# Your code here

## Summary

In this lab we learned that:

- sensors may measure only part of the state;
- observability determines whether hidden states can be reconstructed;
- the observability matrix provides a rank test;
- depth measurements can reveal vertical velocity through the dynamics;
- a Luenberger observer combines prediction and correction;
- observer error evolves according to $A-LC$;
- faster observers converge more quickly but can amplify noise;
- observer innovation is useful for correction and fault detection;
- estimated-state feedback leads naturally to output-feedback control;
- the Kalman filter extends the observer idea to stochastic uncertainty.

### Core insight

> **You do not need to measure every state directly — but the dynamics and sensors must reveal enough information to reconstruct it.**

### Next

Notebook 08 can implement the **Kalman filter** explicitly and study the tradeoff between process uncertainty and sensor uncertainty.